<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_topical_compling_questions/blob/main/%D0%9A%D0%B0%D1%80%D1%85%D1%83%D1%82%D0%9C__module1_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [1]:
!mkdir -p data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
!unzip -o data/lenta.txt.zip -d data/

--2026-10-29 16:02:42--  https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip [following]
--2026-10-29 16:02:43--  https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5723675 (5.5M) [application/zip]
Saving to: ‘data/lenta.txt.zip’

lenta.txt.zip       100%[===================>]   5.46M  --.-KB/s    in 0.07s   

2026-10-29 16:02:43 (75.8 MB/s) - ‘data/lenta.txt.zip’ sa

In [2]:
text = open('data/lenta.txt', encoding='utf-8').read()

In [3]:
import re
from collections import Counter

pattern = re.compile(r'\b(?=[^\s]*[А-Яа-яЁё])(?=[^\s]*[A-Za-z])[A-Za-zА-Яа-яЁё.]+')
Counter(pattern.findall(text)).most_common(10)

[('web', 122),
 ('Яndex', 32),
 ('DVD', 20),
 ('HTML', 13),
 ('IP', 12),
 ('DDoS', 9),
 ('WAP', 8),
 ('cообщает', 7),
 ('МI', 7),
 ('wap', 7)]

Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) примеры слов, в которых совмещается кириллица и латиница. Найдите самое частотное такое слово и вставьте его в форму как ответ.

### Задание 2 (1 балл)

Дана последовательность слов:

`об этом сообщает агентство интерфакс со ссылкой на пресс-службу`

Приведите весь текст корпуса `data/lenta.txt` к нижнему регистру и сделайте простую токенизацию: разбейте все по пробелам, удалите пунктуацию по краям токенов (`strip`) и отфильтруйте пустые строки, оставшиеся после этих шагов.

По частотам из корпуса рассчитайте для каждого триграмма в последовательности условную вероятность P(w3 | w1 w2) = count(w1 w2 w3) / count(w1 w2). Вставьте в форму триграмм с самой низкой вероятностью (три слова через пробел).

*если где-то получается деление но ноль, то пропустите это сочетание

In [4]:
import string

In [7]:
def strip_edge_punct(token: str) -> str:
  return token.strip(string.punctuation)

text = text.lower()

tokens = []
for raw_token in text.split():
  token = strip_edge_punct(raw_token)
  if token:
    tokens.append(token)

bigrams = Counter()
trigrams = Counter()

for i in range(len(tokens) - 1):
  bigrams[(tokens[i], tokens[i + 1])] += 1

for i in range(len(tokens) - 2):
  trigrams[(tokens[i], tokens[i + 1], tokens[i + 2])] += 1

sequence = "об этом сообщает агентство интерфакс со ссылкой на пресс-службу".lower().split()
sequence = [strip_edge_punct(w) for w in sequence]
sequence = [w for w in sequence if w]

results = []

for i in range(len(sequence) - 2):
  w1, w2, w3 = sequence[i], sequence[i + 1], sequence[i + 2]
  count_bigram = bigrams[(w1, w2)]

  if count_bigram == 0:
    continue

  count_trigram = trigrams[(w1, w2, w3)]
  probability = count_trigram / count_bigram

  results.append(
        (
            probability,
            f"{w1} {w2} {w3}",
            count_trigram,
            count_bigram
        )
  )


if results:
  best = min(results, key=lambda x: x[0])
  print("\nТриграмм с минимальной вероятностью:")
  print(best[1])
else:
  print("Нет подходящих триграмм")



Триграмм с минимальной вероятностью:
сообщает агентство интерфакс


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=8)`.

Посчитайте эмбеддинги этих текстов моделью `cointegrated/rubert-tiny2` (без префиксов). Рассчитайте косинусную близость всех со всеми (для каждой пары текстов) и найдите самую близкую пару текстов.

Вставьте в форму первые слова обоих текстов через пробел (например, `Холдинг Американская`, порядок не важен).

!NB: Тексты должны быть разными

In [13]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
lenta = pd.read_csv("https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip").sample(500, random_state=8)

In [16]:
texts = lenta["text"].astype(str).tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2").to(device)
model.eval()

def embed_texts(texts, batch_size=32, max_length=512):
  all_embs = []
  with torch.no_grad():
    for i in range(0, len(texts), batch_size):
      batch = texts[i:i + batch_size]
      enc = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
      ).to(device)
      out = model(**enc)
      emb = out.last_hidden_state[:, 0, :]
      all_embs.append(emb.cpu().numpy())
  return np.vstack(all_embs)

doc_emb = embed_texts(texts)

sims = cosine_similarity(doc_emb, doc_emb)

np.fill_diagonal(sims, -np.inf)

i, j = np.unravel_index(np.argmax(sims), sims.shape)
i, j = int(i), int(j)

print(f"Максимальная косинусная близость: {sims[i, j]:.6f}")
print(f"Индексы в подвыборке: {i}, {j}\n")
print("Текст 1:")
print(texts[i][:50])
print("\n")
print("Текст 2:")
print(texts[j][:50])

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Максимальная косинусная близость: 0.953815
Индексы в подвыборке: 301, 391

Текст 1:
Нападающий «Бастии» Брандао появился на заседании 


Текст 2:
Футболист французского клуба «Бастия» Брандао, сло
